# Liveability Scoring System - EDA and Feature Engineering

This notebook explores the features in `mart_ward_features`, handles missing values, visualizes distributions and correlations, and computes Moran's I spatial autocorrelation for Bengaluru wards.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import libpysal
import esda
from splot.esda import plot_moran, moran_scatterplot
from shapely import wkt

import sys
import os
sys.path.insert(0, os.path.abspath('..'))
from scripts.db_utils import get_db_connection

plt.style.use('ggplot')

## 1. Load Data

We load `mart_ward_features` from PostgreSQL.

In [ ]:
try:
    with get_db_connection() as conn:
        df = pd.read_sql("SELECT * FROM marts.mart_ward_features", conn)
        # The geom comes as WKB, we convert to geopandas later if needed
    print(f"Loaded {len(df)} rows from PostgreSQL.")
except Exception as e:
    print(f"Could not load from DB: {e}")
    # Fallback to CSV
    df = pd.read_csv('../data/processed/ward_features_enriched.csv')
    print(f"Loaded {len(df)} rows from CSV.")

df.head()

## 2. Missingness Heatmap

In [ ]:
plt.figure(figsize=(15, 8))
sns.heatmap(df.isnull(), cbar=False, cmap='viridis')
plt.title('Missingness Heatmap (Yellow = Missing)')
plt.show()

## 3. Correlation Matrix

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Exclude IDs
numeric_cols = [c for c in numeric_cols if c not in ['ward_id', 'ward_number', 'year', 'cluster_id']]

corr = df[numeric_cols].corr()
plt.figure(figsize=(20, 16))
sns.heatmap(corr, cmap='coolwarm', center=0, annot=False, linewidths=.5)
plt.title('Feature Correlation Matrix')
plt.show()

## 4. Feature Distributions

In [ ]:
features_to_plot = ['crime_rate_per_1000', 'avg_aqi', 'avg_ndvi', 'resolution_rate', 'median_price_sqft']

fig, axes = plt.subplots(len(features_to_plot), 1, figsize=(10, 4*len(features_to_plot)))
for i, col in enumerate(features_to_plot):
    if col in df.columns:
        sns.histplot(df[col].dropna(), kde=True, ax=axes[i], color='teal')
        axes[i].set_title(f'Distribution of {col}')
plt.tight_layout()
plt.show()

## 5. Spatial Autocorrelation (Moran's I)

In [ ]:
def run_moran(df, latest_year):
    # We need geometry.
    if 'geom' not in df.columns:
        print("No geometry found, skipping spatial autocorrelation.")
        return
        
    df_latest = df[df['year'] == latest_year].copy()
    # Convert to GeoDataFrame
    df_latest['geometry'] = gpd.GeoSeries.from_wkb(df_latest['geom'], crs="EPSG:4326")
    gdf = gpd.GeoDataFrame(df_latest, geometry='geometry')
    
    # Impute NaNs for target features
    for col in ['crime_rate_per_1000', 'avg_aqi', 'avg_ndvi']:
        gdf[col] = gdf[col].fillna(gdf[col].median())
        
    # Compute spatial weights (Queen contiguity)
    w = libpysal.weights.Queen.from_dataframe(gdf)
    w.transform = 'r'
    
    for col in ['crime_rate_per_1000', 'avg_aqi', 'avg_ndvi']:
        y = gdf[col].values
        moran = esda.Moran(y, w)
        print(f"Global Moran's I for {col}: {moran.I:.3f} (p-value: {moran.p_sim:.3f})")
        
        # Plot
        fig, ax = plt.subplots(figsize=(6, 6))
        moran_scatterplot(moran, ax=ax)
        ax.set_xlabel(col)
        ax.set_ylabel(f'Spatial Lag of {col}')
        plt.title(f"Moran Scatterplot - {col}")
        plt.show()

if 'year' in df.columns:
    run_moran(df, df['year'].max())
